<a href="https://colab.research.google.com/github/Sharmasom/MCP-Powered-Intelligent-Research-Assistant/blob/main/Copy_of_MCP_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "mcp[cli]" google-genai pymupdf streamlit

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("API key configured:", bool(GEMINI_API_KEY))

API key configured: True


In [ ]:
!pip install -q "mcp[cli]" google-genai pymupdf streamlit scikit-learn

In [ ]:
import mcp
import google.genai
import fitz
import streamlit
import sklearn

print("All libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


All libraries imported successfully!


In [ ]:
import pymupdf

print("PyMuPDF imported successfully!")

PyMuPDF imported successfully!


In [ ]:
import fitz

In [ ]:
def search_documents(query):
    documents = [
        "Deepfake detection using EfficientNet",
        "Brain tumor classification using ResNet50",
        "Concrete crack detection using CNN"
    ]

    results = []

    for document in documents:
        if query.lower() in document.lower():
            results.append(document)

    return results


print(search_documents("deepfake"))

['Deepfake detection using EfficientNet']


In [ ]:
import os

PROJECT_DIR = "/content/mcp_research_assistant"
DOCUMENTS_DIR = os.path.join(PROJECT_DIR, "documents")

os.makedirs(DOCUMENTS_DIR, exist_ok=True)

print("Project folder created!")
print("Documents folder:", DOCUMENTS_DIR)

Project folder created!
Documents folder: /content/mcp_research_assistant/documents


In [ ]:
!pip install -q "mcp[cli]<2"

In [ ]:
import os
import pymupdf
from mcp.server.fastmcp import FastMCP

# Create MCP server
mcp = FastMCP("Research Document Server")


@mcp.tool()
def list_documents() -> list[str]:
    """List all available PDF research papers."""

    return [
        filename
        for filename in os.listdir(DOCUMENTS_DIR)
        if filename.lower().endswith(".pdf")
    ]


@mcp.tool()
def read_document(filename: str) -> str:
    """Read and extract text from a selected PDF."""

    # Validate filename
    if (
        not filename.lower().endswith(".pdf")
        or os.path.basename(filename) != filename
    ):
        return "Invalid PDF filename."

    file_path = os.path.join(DOCUMENTS_DIR, filename)

    if not os.path.isfile(file_path):
        return "PDF document not found."

    with pymupdf.open(file_path) as pdf:
        text = "\n".join(page.get_text() for page in pdf)

    return text[:12000]


print("MCP server and two improved document tools registered!")

MCP server and two improved document tools registered!


In [ ]:
@mcp.tool()
def search_documents(query: str) -> str:
    """Search all PDFs and return the top 5 relevant documents."""

    if not query.strip():
        return "Please enter a search query."

    documents = []

    # Extract text from all PDFs
    for filename in list_documents():

        file_path = os.path.join(DOCUMENTS_DIR, filename)

        try:
            with pymupdf.open(file_path) as pdf:
                text = "\n".join(
                    page.get_text() for page in pdf
                )

            if text.strip():
                documents.append((filename, text))

        except Exception as error:
            print(f"Could not read {filename}: {error}")

    if not documents:
        return "No readable PDF documents found."

    # Convert documents into TF-IDF vectors
    texts = [doc[1] for doc in documents]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=20000
    )

    document_vectors = vectorizer.fit_transform(texts)

    # Convert query into a vector
    query_vector = vectorizer.transform([query])

    # Calculate similarity scores
    scores = cosine_similarity(
        query_vector, document_vectors
    ).flatten()

    # Sort by relevance
    ranked_indices = scores.argsort()[::-1]

    results = []

    for index in ranked_indices[:5]:

        if scores[index] <= 0:
            continue

        filename, content = documents[index]

        results.append(
            f"\nPaper: {filename}\n"
            f"Similarity Score: {scores[index]:.3f}\n"
            f"Content:\n{content[:2500]}\n"
        )

    return "\n".join(results) if results else "No relevant documents found."


print("Updated search_documents tool registered!")

Updated search_documents tool registered!


In [ ]:
tools = mcp._tool_manager.list_tools()

for tool in tools:
    print(tool.name)

print("Total tools:", len(tools))

list_documents
read_document
search_documents
Total tools: 3


In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        destination = os.path.join(DOCUMENTS_DIR, filename)
        shutil.move(filename, destination)
        print(f"Uploaded: {filename}")
    else:
        print(f"Skipped (not a PDF): {filename}")

Saving Brain_tumor2.pdf to Brain_tumor2.pdf
Saving Brain_tuomr1.pdf to Brain_tuomr1.pdf
Saving Deepfake.pdf to Deepfake.pdf
Saving house_prediction.pdf to house_prediction.pdf
Saving llm.pdf to llm.pdf
Uploaded: Brain_tumor2.pdf
Uploaded: Brain_tuomr1.pdf
Uploaded: Deepfake.pdf
Uploaded: house_prediction.pdf
Uploaded: llm.pdf


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("TF-IDF libraries imported successfully!")

TF-IDF libraries imported successfully!


In [ ]:
result = search_documents("deep learning models")
print(result)


Paper: Brain_tuomr1.pdf
Similarity Score: 0.193
Content:
Citation: Abdusalomov, A.B.;
Mukhiddinov, M.; Whangbo, T.K.
Brain Tumor Detection Based on
Deep Learning Approaches and
Magnetic Resonance Imaging.
Cancers 2023, 15, 4172. https://
doi.org/10.3390/cancers15164172
Academic Editor: Hirotaka
Hasegawa
Received: 27 June 2023
Revised: 11 August 2023
Accepted: 17 August 2023
Published: 18 August 2023
Copyright:
© 2023 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed
under
the
terms
and
conditions of the Creative Commons
Attribution (CC BY) license (https://
creativecommons.org/licenses/by/
4.0/).
cancers
Article
Brain Tumor Detection Based on Deep Learning Approaches
and Magnetic Resonance Imaging
Akmalbek Bobomirzaevich Abdusalomov *
, Mukhriddin Mukhiddinov
and Taeg Keun Whangbo *
Department of Computer Engineering, Gachon University, Seongnam-si 13120, Republic of Korea;
mukhiddinov18@gachon.ac.kr
* Correspondence: bobomirzaevich@

In [ ]:
tools = mcp._tool_manager.list_tools()

print("Registered MCP tools:")

for tool in tools:
    print("-", tool.name)

print("\nTotal tools:", len(tools))

Registered MCP tools:
- list_documents
- read_document
- search_documents

Total tools: 3


In [ ]:
tools = mcp._tool_manager.list_tools()

print("Registered MCP tools:")

for tool in tools:
    print("-", tool.name)

print("\nTotal tools:", len(tools))

Registered MCP tools:
- list_documents
- read_document
- search_documents

Total tools: 3


In [ ]:
%%writefile /content/mcp_research_assistant/server.py

import os
import pymupdf

from mcp.server.fastmcp import FastMCP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DOCUMENTS_DIR = "/content/mcp_research_assistant/documents"

# Create the MCP server
mcp = FastMCP("Research Document Server")


def get_pdf_files():
    """Return the names of all PDFs in the documents folder."""
    return [
        filename
        for filename in os.listdir(DOCUMENTS_DIR)
        if filename.lower().endswith(".pdf")
    ]


@mcp.tool()
def list_documents() -> list[str]:
    """List all available research PDFs."""
    return get_pdf_files()


@mcp.tool()
def read_document(filename: str) -> str:
    """Extract text from a selected PDF."""
    if filename not in get_pdf_files():
        return "PDF document not found."

    file_path = os.path.join(DOCUMENTS_DIR, filename)
    text = ""

    with pymupdf.open(file_path) as pdf:
        for page in pdf:
            text += page.get_text() + "\n"

    return text[:12000]


@mcp.tool()
def search_documents(query: str) -> str:
    """Search all PDFs and return the five most relevant documents."""

    if not query.strip():
        return "Please enter a search query."

    documents = []

    for filename in get_pdf_files():
        file_path = os.path.join(DOCUMENTS_DIR, filename)

        try:
            with pymupdf.open(file_path) as pdf:
                content = "\n".join(page.get_text() for page in pdf)

            if content.strip():
                documents.append((filename, content))

        except Exception:
            continue

    if not documents:
        return "No readable PDF documents found."

    texts = [content for _, content in documents]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=20000
    )

    document_vectors = vectorizer.fit_transform(texts)
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector, document_vectors
    ).flatten()

    ranked_indices = scores.argsort()[::-1]

    results = []

    for index in ranked_indices[:5]:
        if scores[index] <= 0:
            continue

        filename, content = documents[index]

        results.append(
            f"Paper: {filename}\n"
            f"Similarity Score: {scores[index]:.3f}\n"
            f"Content:\n{content[:2500]}\n"
        )

    return "\n".join(results) if results else "No relevant documents found."


if __name__ == "__main__":
    # STDIO is used for communication with the MCP client.
    mcp.run(transport="stdio")

Writing /content/mcp_research_assistant/server.py


In [ ]:
import asyncio
import sys
import os

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Location of our MCP server
SERVER_PATH = "/content/mcp_research_assistant/server.py"

# Configure how the client launches the server
server_params = StdioServerParameters(
    command=sys.executable,
    args=[SERVER_PATH],
    env=os.environ.copy()
)

print("MCP client configuration completed!")

MCP client configuration completed!


In [ ]:
import sys

async def test_mcp_connection():

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            print("MCP connection established!")

            response = await session.list_tools()

            print("\nTools discovered through MCP:")

            for tool in response.tools:
                print("-", tool.name)

            result = await session.call_tool(
                "list_documents",
                arguments={}
            )

            print("\nPDF documents retrieved through MCP:")

            for item in result.content:
                if item.type == "text":
                    print(item.text)


await test_mcp_connection()

MCP connection established!

Tools discovered through MCP:
- list_documents
- read_document
- search_documents

PDF documents retrieved through MCP:
Deepfake.pdf
house_prediction.pdf
llm.pdf
Brain_tumor2.pdf
Brain_tuomr1.pdf


In [ ]:
async def test_remaining_tools():

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            # Test 1: Read a PDF
            print("TEST 1: Reading a PDF")

            read_result = await session.call_tool(
                "read_document",
                arguments={"filename": "Deepfake.pdf"}
            )

            for item in read_result.content:
                if item.type == "text":
                    print(item.text[:1000])

            # Test 2: Search all PDFs
            print("\nTEST 2: Searching PDFs")

            search_result = await session.call_tool(
                "search_documents",
                arguments={"query": "deep learning models"}
            )

            for item in search_result.content:
                if item.type == "text":
                    print(item.text[:2000])


await test_remaining_tools()

TEST 1: Reading a PDF
Vol.:(0123456789)
Artificial Intelligence Review (2024) 57:159
https://doi.org/10.1007/s10462-024-10810-6
1 3
Deepfake video detection: challenges and opportunities
Achhardeep Kaur1 · Azadeh Noori Hoshyar2 · Vidya Saikrishna3 · Selena Firmin1 · 
Feng Xia4
Accepted: 16 May 2024 / Published online: 29 May 2024 
© The Author(s) 2024
Abstract
Deepfake videos are a growing social issue. These videos are manipulated by artificial 
intelligence (AI) techniques (especially deep learning), an emerging societal issue. Mali-
cious individuals misuse deepfake technologies to spread false information, such as fake 
images, videos, and audio. The development of convincing fake content threatens politics, 
security, and privacy. The majority of deepfake video detection methods are data-driven. 
This survey paper aims to thoroughly analyse deepfake video generation and detection. The 
paper’s main contribution is the classification of the many challenges encountered while 
detect

In [ ]:
from google import genai
from google.colab import userdata

# Retrieve API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# Initialise Gemini client
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

# Select the Gemini model
MODEL_NAME = "gemini-3.5-flash-
lite"

print("Gemini client initialised successfully!")

Gemini client initialised successfully!


In [ ]:
print("Current model:", repr(MODEL_NAME))

print("\nChecking models available to your API key...\n")

available_models = gemini_client.models.list()

for model in available_models:
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

Current model: 'gemini-3.5 Flash-Lite'

Checking models available to your API key...

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-

In [ ]:
from google import genai
from google.colab import userdata

# Initialise Gemini
gemini_client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# Use the model recommended by Google's API
MODEL_NAME = "gemini-3.1-flash-lite"

# Test Gemini
response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents="Say hello in one sentence."
)

print(response.text)

Hello, it is a pleasure to meet you!


In [ ]:
async def summarize_paper(filename):

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            # Step 1: Read PDF through MCP
            result = await session.call_tool(
                "read_document",
                arguments={"filename": filename}
            )

            paper_text = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

            # Step 2: Ask Gemini to summarise
            prompt = f"""
            You are an academic research assistant.

            Summarise the following research paper.

            Include:
            1. Research objective
            2. Proposed methodology
            3. Models and algorithms
            4. Dataset used
            5. Experimental results
            6. Limitations
            7. Future research directions

            Important:
            - Use only information from the paper.
            - Do not invent results or metrics.
            - Write "Not reported" if information is unavailable.
            - Use simple English.

            Research paper:
            {paper_text}
            """

            response = gemini_client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            return response.text


summary = await summarize_paper("Deepfake.pdf")

print(summary)

This research paper is a survey article that provides an analysis of deepfake video generation and the current state of detection technologies.

**1. Research objective**
To provide a comprehensive analysis of deepfake video generation and detection, specifically focusing on categorizing the challenges faced by current detection methods and identifying research gaps to guide future studies.

**2. Proposed methodology**
The study adopts a survey-based review approach. It critically evaluates existing literature to classify challenges in deepfake detection, including issues related to data (e.g., unbalanced datasets), training (e.g., computational resource requirements), and reliability (e.g., overconfidence in models and emerging manipulation techniques).

**3. Models and algorithms**
The paper emphasizes that deep learning-based methods currently dominate the field of deepfake detection. It highlights the use of Generative Adversarial Networks (GANs) as both a tool for creating sophist

In [ ]:
SERVER_PATH = "/content/mcp_research_assistant/server.py"

with open(SERVER_PATH, "r") as file:
    server_code = file.read()

server_code = server_code.replace(
    "return text[:12000]",
    "return text"
)

with open(SERVER_PATH, "w") as file:
    file.write(server_code)

print("PDF text limit removed successfully!")

PDF text limit removed successfully!


In [ ]:
with open(SERVER_PATH, "r") as file:
    code = file.read()

print("Unlimited text extraction:", "return text[:12000]" not in code)

Unlimited text extraction: True


In [ ]:
def split_into_chunks(text, chunk_size=12000, overlap=500):
    """
    Divide long PDF text into smaller chunks.
    """

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


print("Chunking function created successfully!")

Chunking function created successfully!


In [ ]:
sample_text = "Research paper content. " * 3000

chunks = split_into_chunks(sample_text)

print("Total characters:", len(sample_text))
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk)} characters")

Total characters: 72000
Total chunks: 7
Chunk 1: 12000 characters
Chunk 2: 12000 characters
Chunk 3: 12000 characters
Chunk 4: 12000 characters
Chunk 5: 12000 characters
Chunk 6: 12000 characters
Chunk 7: 3000 characters


In [ ]:
def summarize_long_text(text):

    chunks = split_into_chunks(text)

    chunk_summaries = []

    print("Total chunks:", len(chunks))

    # MAP: Summarise each chunk
    for i, chunk in enumerate(chunks):

        print(f"Summarising chunk {i+1}/{len(chunks)}...")

        prompt = f"""
        You are an academic research assistant.

        Summarise this section of a research paper.
        Preserve important technical details.
        Do not invent information.

        Section:
        {chunk}
        """

        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        chunk_summaries.append(response.text)

    # REDUCE: Combine individual summaries
    print("\nCombining summaries...")

    final_prompt = f"""
    You are an academic research assistant.

    Combine the following section summaries into
    one comprehensive research paper summary.

    Use these headings:
    1. Research Objective
    2. Methodology
    3. Models and Algorithms
    4. Dataset
    5. Experimental Results
    6. Limitations
    7. Future Research Directions

    Do not invent missing information.
    If something is not reported, say so.

    Section summaries:
    {"".join(chunk_summaries)}
    """

    final_response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=final_prompt
    )

    return final_response.text


print("Map-Reduce summarisation function created!")

Map-Reduce summarisation function created!


In [ ]:
paper_result = await summarize_paper("Deepfake.pdf")

print("PDF text retrieved successfully!")

PDF text retrieved successfully!


In [ ]:
def summarize_long_text(text):

    # Divide the document into chunks
    chunks = split_into_chunks(
        text,
        chunk_size=12000,
        overlap=500
    )

    chunk_summaries = []

    print("Total chunks:", len(chunks))

    # MAP PHASE: Summarise each chunk
    for i, chunk in enumerate(chunks):

        print(f"Summarising chunk {i+1}/{len(chunks)}...")

        prompt = f"""
        You are an academic research assistant.

        Summarise the following section of a research paper.

        Extract:
        - Important concepts
        - Research methodology
        - Models and algorithms
        - Dataset information
        - Experimental findings
        - Limitations
        - Future research directions

        Instructions:
        - Preserve important technical details.
        - Do not invent facts or numerical results.
        - If information is unavailable, do not assume it.

        PAPER SECTION:
        {chunk}
        """

        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        chunk_summaries.append(response.text)

    # REDUCE PHASE: Combine all chunk summaries
    print("\nCombining chunk summaries...")

    final_prompt = f"""
    You are an academic research assistant.

    Create one comprehensive summary of this research paper
    using the following individual section summaries.

    Use these headings:

    1. Research Objective
    2. Proposed Methodology
    3. Models and Algorithms
    4. Dataset Used
    5. Experimental Results
    6. Limitations
    7. Future Research Directions

    Important instructions:
    - Combine repeated information.
    - Preserve important technical details.
    - Do not invent datasets, metrics or results.
    - Clearly distinguish survey papers from experimental papers.
    - Write "Not reported" when information is unavailable.
    - Use simple, clear English.

    SECTION SUMMARIES:

    {"".join(chunk_summaries)}
    """

    final_response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=final_prompt
    )

    return final_response.text


# Generate final research paper summary
final_summary = summarize_long_text(paper_text)

print("\n========== RESEARCH PAPER SUMMARY ==========\n")
print(final_summary)

NameError: name 'paper_text' is not defined

In [ ]:
async def get_paper_text(filename):

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "read_document",
                arguments={"filename": filename}
            )

            paper_text = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

            return paper_text


# Retrieve PDF text
paper_text = await get_paper_text("Deepfake.pdf")

print("PDF text retrieved successfully!")
print("Total characters:", len(paper_text))

PDF text retrieved successfully!
Total characters: 139480


In [ ]:
def summarize_long_text(text):

    # Divide the document into chunks
    chunks = split_into_chunks(
        text,
        chunk_size=12000,
        overlap=500
    )

    chunk_summaries = []

    print("Total chunks:", len(chunks))

    # MAP PHASE: Summarise each chunk
    for i, chunk in enumerate(chunks):

        print(f"Summarising chunk {i+1}/{len(chunks)}...")

        prompt = f"""
        You are an academic research assistant.

        Summarise the following section of a research paper.

        Extract:
        - Important concepts
        - Research methodology
        - Models and algorithms
        - Dataset information
        - Experimental findings
        - Limitations
        - Future research directions

        Instructions:
        - Preserve important technical details.
        - Do not invent facts or numerical results.
        - If information is unavailable, do not assume it.

        PAPER SECTION:
        {chunk}
        """

        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        chunk_summaries.append(response.text)

    # REDUCE PHASE: Combine all chunk summaries
    print("\nCombining chunk summaries...")

    final_prompt = f"""
    You are an academic research assistant.

    Create one comprehensive summary of this research paper
    using the following individual section summaries.

    Use these headings:

    1. Research Objective
    2. Proposed Methodology
    3. Models and Algorithms
    4. Dataset Used
    5. Experimental Results
    6. Limitations
    7. Future Research Directions

    Important instructions:
    - Combine repeated information.
    - Preserve important technical details.
    - Do not invent datasets, metrics or results.
    - Clearly distinguish survey papers from experimental papers.
    - Write "Not reported" when information is unavailable.
    - Use simple, clear English.

    SECTION SUMMARIES:

    {"".join(chunk_summaries)}
    """

    final_response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=final_prompt
    )

    return final_response.text


# Generate final research paper summary
final_summary = summarize_long_text(paper_text)

print("\n========== RESEARCH PAPER SUMMARY ==========\n")
print(final_summary)

Total chunks: 13
Summarising chunk 1/13...
Summarising chunk 2/13...
Summarising chunk 3/13...
Summarising chunk 4/13...
Summarising chunk 5/13...
Summarising chunk 6/13...
Summarising chunk 7/13...
Summarising chunk 8/13...
Summarising chunk 9/13...
Summarising chunk 10/13...
Summarising chunk 11/13...
Summarising chunk 12/13...
Summarising chunk 13/13...

Combining chunk summaries...

========== RESEARCH PAPER SUMMARY ==========

This comprehensive summary synthesizes the provided research paper, **"Deepfake video detection: challenges and opportunities" by Kaur et al. (2024)**, which acts as a systematic literature review (SLR) evaluating the current state of deepfake generation and the corresponding efforts to combat them.

### 1. Research Objective
The study aims to provide a comprehensive analysis of the "arms race" between deepfake generation techniques and detection methods. The authors seek to categorize existing literature, identify current research gaps—specifically regardin

In [ ]:
SERVER_PATH = "/content/mcp_research_assistant/server.py"

with open(SERVER_PATH, "r") as file:
    server_code = file.read()

new_tool = '''

@mcp.tool()
def read_document_with_pages(filename: str) -> str:
    """Extract PDF text with page numbers for source verification."""

    if filename not in get_pdf_files():
        return "PDF document not found."

    file_path = os.path.join(DOCUMENTS_DIR, filename)
    page_texts = []

    with pymupdf.open(file_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            page_text = page.get_text()

            if page_text.strip():
                page_texts.append(
                    f"--- PAGE {page_number} ---\\\\n{page_text}"
                )

    return "\\\\n".join(page_texts)
'''

if "def read_document_with_pages" not in server_code:
    server_code = server_code.replace(
        'if __name__ == "__main__":',
        new_tool + '\\n\\nif __name__ == "__main__":'
    )

    with open(SERVER_PATH, "w") as file:
        file.write(server_code)

    print("Page-wise extraction tool added!")
else:
    print("Page-wise extraction tool already exists!")

Page-wise extraction tool added!


In [ ]:
async def test_page_extraction():
    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()

            print("Available MCP tools:")
            for tool in tools.tools:
                print("-", tool.name)

            result = await session.call_tool(
                "read_document_with_pages",
                arguments={"filename": "Deepfake.pdf"}
            )

            extracted_text = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

            print("\nExtraction successful!")
            print(extracted_text[:2000])

await test_page_extraction()

ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

In [ ]:
import traceback

async def test_page_extraction_debug():
    try:
        async with stdio_client(
            server_params,
            errlog=sys.__stderr__
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                print("MCP connection established!")

                tools = await session.list_tools()
                print("\nAvailable tools:")
                for tool in tools.tools:
                    print("-", tool.name)

                result = await session.call_tool(
                    "read_document_with_pages",
                    arguments={"filename": "Deepfake.pdf"}
                )

                extracted_text = "\n".join(
                    item.text
                    for item in result.content
                    if item.type == "text"
                )

                print("\nExtraction successful!")
                print(extracted_text[:2000])

    except BaseExceptionGroup as error_group:
        print("Underlying error:")
        for error in error_group.exceptions:
            traceback.print_exception(error)

    except Exception:
        traceback.print_exc()

await test_page_extraction_debug()

Underlying error:


  + Exception Group Traceback (most recent call last):
  |   File "/usr/local/lib/python3.13/dist-packages/mcp/client/stdio/__init__.py", line 189, in stdio_client
  |     yield read_stream, write_stream
  |   File "/tmp/ipykernel_19686/3369823321.py", line 10, in test_page_extraction_debug
  |     async with ClientSession(read, write) as session:
  |                ~~~~~~~~~~~~~^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.13/dist-packages/mcp/shared/session.py", line 238, in __aexit__
  |     return await self._task_group.__aexit__(exc_type, exc_val, exc_tb)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.13/dist-packages/anyio/_backends/_asyncio.py", line 871, in __aexit__
  |     raise BaseExceptionGroup(
  |         "unhandled errors in a TaskGroup", self._exceptions
  |     ) from None
  | ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most

In [ ]:
import os
import py_compile

SERVER_PATH = "/content/mcp_research_assistant/server.py"

print("Server file exists:", os.path.exists(SERVER_PATH))

try:
    py_compile.compile(SERVER_PATH, doraise=True)
    print("Syntax check: PASSED")
except Exception as error:
    print("Syntax check: FAILED")
    print(error)

print("\nLast 70 lines of server.py:\n")

with open(SERVER_PATH, "r") as file:
    lines = file.readlines()

for line_number, line in enumerate(lines[-70:], start=max(1, len(lines)-69)):
    print(f"{line_number}: {line}", end="")

Server file exists: True
Syntax check: FAILED
  File "/content/mcp_research_assistant/server.py", line 126
    \n\nif __name__ == "__main__":
     ^
SyntaxError: unexpected character after line continuation character


Last 70 lines of server.py:

59:             with pymupdf.open(file_path) as pdf:
60:                 content = "\n".join(page.get_text() for page in pdf)
61: 
62:             if content.strip():
63:                 documents.append((filename, content))
64: 
65:         except Exception:
66:             continue
67: 
68:     if not documents:
69:         return "No readable PDF documents found."
70: 
71:     texts = [content for _, content in documents]
72: 
73:     vectorizer = TfidfVectorizer(
74:         stop_words="english",
75:         max_features=20000
76:     )
77: 
78:     document_vectors = vectorizer.fit_transform(texts)
79:     query_vector = vectorizer.transform([query])
80: 
81:     scores = cosine_similarity(
82:         query_vector, document_vectors
83: 

In [ ]:
SERVER_PATH = "/content/mcp_research_assistant/server.py"

with open(SERVER_PATH, "r") as file:
    server_code = file.read()

# Fix the incorrectly inserted literal line breaks
server_code = server_code.replace(
    r'\n\nif __name__ == "__main__":',
    '\n\nif __name__ == "__main__":'
)

# Fix escaped newlines in the page extraction tool
server_code = server_code.replace(r'\\n', r'\n')

with open(SERVER_PATH, "w") as file:
    file.write(server_code)

# Verify Python syntax
import py_compile

try:
    py_compile.compile(SERVER_PATH, doraise=True)
    print("Server syntax check: PASSED!")
except Exception as error:
    print("Server syntax check: FAILED")
    print(error)

Server syntax check: PASSED!


In [ ]:
await test_page_extraction_debug()

MCP connection established!

Available tools:
- list_documents
- read_document
- search_documents
- read_document_with_pages

Extraction successful!
--- PAGE 1 ---
Vol.:(0123456789)
Artificial Intelligence Review (2024) 57:159
https://doi.org/10.1007/s10462-024-10810-6
1 3
Deepfake video detection: challenges and opportunities
Achhardeep Kaur1 · Azadeh Noori Hoshyar2 · Vidya Saikrishna3 · Selena Firmin1 · 
Feng Xia4
Accepted: 16 May 2024 / Published online: 29 May 2024 
© The Author(s) 2024
Abstract
Deepfake videos are a growing social issue. These videos are manipulated by artificial 
intelligence (AI) techniques (especially deep learning), an emerging societal issue. Mali-
cious individuals misuse deepfake technologies to spread false information, such as fake 
images, videos, and audio. The development of convincing fake content threatens politics, 
security, and privacy. The majority of deepfake video detection methods are data-driven. 
This survey paper aims to thoroughly analyse 

In [ ]:
async def verify_summary(filename, summary):

    # Connect to MCP server
    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            # Retrieve original PDF with page numbers
            result = await session.call_tool(
                "read_document_with_pages",
                arguments={"filename": filename}
            )

            paper_text = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

    # Ask Gemini to verify the summary
    prompt = f"""
You are an academic research paper fact-checking assistant.

Your task is to verify the generated summary against the original
research paper.

ORIGINAL RESEARCH PAPER:
{paper_text}

GENERATED SUMMARY:
{summary}

Instructions:

1. Break the summary into individual factual claims.
2. Check every claim against the original paper.
3. Assign one status to each claim:

   SUPPORTED: Directly supported by the paper.
   PARTIALLY SUPPORTED: Only some parts are supported.
   NOT FOUND: No supporting evidence was located.
   CONTRADICTED: The paper explicitly states something different.

4. Provide the page number for supporting evidence.
5. Include a short evidence quotation from the paper.
6. Pay special attention to:
   - Numerical claims and statistics.
   - Dataset names.
   - Model names and algorithms.
   - Experimental results and accuracy values.
   - Number of studies reviewed.
7. Do not treat information missing from the paper as fact.
8. Do not invent page numbers or evidence.
9. If evidence is insufficient, clearly say so.

OUTPUT FORMAT:

### Summary Verification

For each claim, provide:

- Claim:
- Status:
- Evidence:
- Page number:
- Explanation:

### Overall Assessment
Explain whether the summary is reliable and identify claims
that should be corrected or removed.

"""

    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text

In [ ]:
verification_report = await verify_summary(
    "Deepfake.pdf",
    summary
)

print(verification_report)

### Summary Verification

**1. Claim:** The research paper is a survey article that provides an analysis of deepfake video generation and the current state of detection technologies.
- **Status:** SUPPORTED
- **Evidence:** "This survey paper aims to thoroughly analyse deepfake video generation and detection."
- **Page number:** 1
- **Explanation:** The abstract clearly states the purpose of the paper is to provide this analysis.

**2. Claim:** The paper's objective is to provide a comprehensive analysis of deepfake video generation and detection, specifically focusing on categorizing the challenges faced by current detection methods and identifying research gaps to guide future studies.
- **Status:** SUPPORTED
- **Evidence:** "The paper’s main contribution is the classification of the many challenges encountered while detecting deepfake videos... The study also indicates major research gaps, guiding future deepfake detection research."
- **Page number:** 1
- **Explanation:** These are 

In [ ]:
import json
import re

async def verify_summary_v2(filename, summary):

    # 1. Retrieve the original PDF through MCP
    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            result = await session.call_tool(
                "read_document_with_pages",
                arguments={"filename": filename}
            )

            paper_text = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

    # 2. Prepare verification instructions
    prompt = f"""
You are an academic research paper fact-checking assistant.

Compare the generated summary against the original research paper.

ORIGINAL PAPER:
{paper_text}

GENERATED SUMMARY:
{summary}

For every factual claim in the summary, provide:

1. claim: The claim being checked.
2. status: SUPPORTED, PARTIALLY SUPPORTED, NOT FOUND, or CONTRADICTED.
3. evidence: A short exact quotation from the original paper.
4. page_number: The original PDF page number(s).
5. evidence_quality: STRONG, MODERATE, or WEAK.
6. explanation: Explain why the evidence supports or fails to support
   the claim.
7. confidence: HIGH, MEDIUM, or LOW.

Verification rules:

- Use only the original paper as evidence.
- Never invent quotations or page numbers.
- Preserve the original meaning of the paper.
- A quotation must directly support the claim.
- If evidence is insufficient, use NOT FOUND.
- Use CONTRADICTED only when the paper explicitly disagrees.
- Do not confuse datasets reviewed by a survey with datasets used
  in new experiments.
- Do not confuse results reported by earlier studies with new
  experimental results from the survey authors.
- Confidence is a qualitative assessment, not a probability.
- If no evidence exists, use an empty evidence string and null
  page_number.

Return ONLY valid JSON in this exact structure:

{{
  "paper_name": "filename",
  "claims": [
    {{
      "claim": "Claim text",
      "status": "SUPPORTED",
      "evidence": "Exact quotation",
      "page_number": [1],
      "evidence_quality": "STRONG",
      "explanation": "Explanation",
      "confidence": "HIGH"
    }}
  ],
  "overall_assessment": "Overall reliability assessment",
  "important_corrections": [
    "Correction 1"
  ]
}}
"""

    # 3. Ask Gemini to verify the claims
    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    # 4. Convert Gemini's JSON response into Python data
    raw_response = response.text.strip()

    raw_response = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_response
    )

    report = json.loads(raw_response)

    return report

In [ ]:
verification_report_v2 = await verify_summary_v2(
    "Deepfake.pdf",
    summary
)

print(json.dumps(
    verification_report_v2,
    indent=2,
    ensure_ascii=False
))

{
  "paper_name": "Artificial Intelligence Review (2024) 57:159",
  "claims": [
    {
      "claim": "The research paper is a survey article providing an analysis of deepfake video generation and detection technologies.",
      "status": "SUPPORTED",
      "evidence": "This survey paper aims to thoroughly analyse deepfake video generation and detection.",
      "page_number": [
        1
      ],
      "evidence_quality": "STRONG",
      "explanation": "The abstract explicitly states the paper's purpose as a survey of deepfake video generation and detection.",
      "confidence": "HIGH"
    },
    {
      "claim": "The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.",
      "status": "SUPPORTED",
      "evidence": "The paper’s main contribution is the classification of the many challenges encountered while detecting deepfake videos. The study also indicates major research gaps, guiding future deepfake detectio

In [ ]:
for claim in verification_report_v2["claims"]:

    print("\n" + "=" * 65)
    print("CLAIM:", claim["claim"])
    print("STATUS:", claim["status"])
    print("EVIDENCE QUALITY:", claim["evidence_quality"])
    print("CONFIDENCE:", claim["confidence"])
    print("PAGE:", claim["page_number"])
    print("EVIDENCE:", claim["evidence"])
    print("EXPLANATION:", claim["explanation"])

print("\nOVERALL ASSESSMENT:")
print(verification_report_v2["overall_assessment"])

print("\nIMPORTANT CORRECTIONS:")
for correction in verification_report_v2["important_corrections"]:
    print("-", correction)


CLAIM: The research paper is a survey article providing an analysis of deepfake video generation and detection technologies.
STATUS: SUPPORTED
EVIDENCE QUALITY: STRONG
CONFIDENCE: HIGH
PAGE: [1]
EVIDENCE: This survey paper aims to thoroughly analyse deepfake video generation and detection.
EXPLANATION: The abstract explicitly states the paper's purpose as a survey of deepfake video generation and detection.

CLAIM: The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.
STATUS: SUPPORTED
EVIDENCE QUALITY: STRONG
CONFIDENCE: HIGH
PAGE: [1]
EVIDENCE: The paper’s main contribution is the classification of the many challenges encountered while detecting deepfake videos. The study also indicates major research gaps, guiding future deepfake detection research.
EXPLANATION: The abstract confirms the goals of classifying challenges and identifying research gaps to guide future work.

CLAIM: The study adopts a survey-base

In [ ]:
def validate_evidence(report, paper_text):

    pages = {}
    current_page = None

    # Separate extracted text by page
    for line in paper_text.splitlines():

        match = re.match(r"--- PAGE (\d+) ---", line)

        if match:
            current_page = int(match.group(1))
            pages[current_page] = ""
        elif current_page is not None:
            pages[current_page] += line + "\n"

    # Validate each claim's evidence
    for claim in report["claims"]:

        evidence = claim.get("evidence", "").strip()
        page_numbers = claim.get("page_number") or []

        claim["evidence_exists"] = False
        claim["page_match"] = False
        claim["validation_status"] = "NEEDS REVIEW"

        if not evidence:
            continue

        # Check whether quotation exists anywhere in the PDF
        normalized_evidence = " ".join(evidence.split()).lower()

        for page_number in page_numbers:

            page_content = pages.get(page_number, "")
            normalized_page = " ".join(page_content.split()).lower()

            if normalized_evidence in normalized_page:
                claim["evidence_exists"] = True
                claim["page_match"] = True
                claim["validation_status"] = "EXACT MATCH"
                break

        # Check whether the quotation exists on any page
        if not claim["evidence_exists"]:

            for page_content in pages.values():

                normalized_page = " ".join(
                    page_content.split()
                ).lower()

                if normalized_evidence in normalized_page:
                    claim["evidence_exists"] = True
                    claim["validation_status"] = (
                        "QUOTE FOUND, PAGE MISMATCH"
                    )
                    break

    return report

In [ ]:
paper_text = await get_paper_text("Deepfake.pdf")

validated_report = validate_evidence(
    verification_report_v2,
    paper_text
)

for claim in validated_report["claims"]:

    print("\nClaim:", claim["claim"])
    print("Gemini status:", claim["status"])
    print("Evidence exists:", claim["evidence_exists"])
    print("Page matches:", claim["page_match"])
    print("Validation:", claim["validation_status"])


Claim: The research paper is a survey article providing an analysis of deepfake video generation and detection technologies.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: The study adopts a survey-based review approach and identifies data, training, and reliability as the main categories of challenges.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: Deep learning-based methods currently dominate deepfake detection, with GANs serving as a key tool for creation and a subject for detection research.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: The paper does not report new experimental results as it i

In [ ]:
async def get_pagewise_paper_text(filename):

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            result = await session.call_tool(
                "read_document_with_pages",
                arguments={"filename": filename}
            )

            return "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )


paper_text = await get_pagewise_paper_text("Deepfake.pdf")

print("Page-wise extraction successful!")
print("Page markers found:", len(re.findall(
    r"--- PAGE \d+ ---", paper_text
)))

print("\nFirst 500 characters:")
print(paper_text[:500])

Page-wise extraction successful!
Page markers found: 47

First 500 characters:
--- PAGE 1 ---
Vol.:(0123456789)
Artificial Intelligence Review (2024) 57:159
https://doi.org/10.1007/s10462-024-10810-6
1 3
Deepfake video detection: challenges and opportunities
Achhardeep Kaur1 · Azadeh Noori Hoshyar2 · Vidya Saikrishna3 · Selena Firmin1 · 
Feng Xia4
Accepted: 16 May 2024 / Published online: 29 May 2024 
© The Author(s) 2024
Abstract
Deepfake videos are a growing social issue. These videos are manipulated by artificial 
intelligence (AI) techniques (especially deep learning),


In [ ]:
validated_report = validate_evidence(
    verification_report_v2,
    paper_text
)

for claim in validated_report["claims"]:

    print("\nClaim:", claim["claim"])
    print("Gemini status:", claim["status"])
    print("Evidence exists:", claim["evidence_exists"])
    print("Page matches:", claim["page_match"])
    print("Validation:", claim["validation_status"])


Claim: The research paper is a survey article providing an analysis of deepfake video generation and detection technologies.
Gemini status: SUPPORTED
Evidence exists: True
Page matches: True
Validation: EXACT MATCH

Claim: The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: The study adopts a survey-based review approach and identifies data, training, and reliability as the main categories of challenges.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: Deep learning-based methods currently dominate deepfake detection, with GANs serving as a key tool for creation and a subject for detection research.
Gemini status: SUPPORTED
Evidence exists: False
Page matches: False
Validation: NEEDS REVIEW

Claim: The paper does not report new experimental results as it is a

In [ ]:
import re
import unicodedata

def normalize_text(text):

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Join words split by PDF line wrapping
    text = re.sub(r"-\s*\n\s*", "", text)

    # Normalize punctuation and whitespace
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)

    return " ".join(text.split())


def validate_evidence_v2(report, paper_text):

    pages = {}
    current_page = None

    # Separate PDF text page by page
    for line in paper_text.splitlines():

        match = re.match(r"--- PAGE (\d+) ---", line)

        if match:
            current_page = int(match.group(1))
            pages[current_page] = ""

        elif current_page is not None:
            pages[current_page] += line + "\n"

    # Normalize every page
    normalized_pages = {
        page: normalize_text(content)
        for page, content in pages.items()
    }

    for claim in report["claims"]:

        evidence = claim.get("evidence", "").strip()

        # Split evidence where Gemini indicates omitted text
        fragments = re.split(
            r"\.{3,}|…",
            evidence
        )

        fragments = [
            normalize_text(fragment)
            for fragment in fragments
            if len(normalize_text(fragment).split()) >= 4
        ]

        matched_fragments = []
        unmatched_fragments = []
        matched_pages = set()

        for fragment in fragments:

            found = False

            for page_number, page_content in normalized_pages.items():

                if fragment in page_content:

                    found = True
                    matched_pages.add(page_number)

            if found:
                matched_fragments.append(fragment)
            else:
                unmatched_fragments.append(fragment)

        # Store validation details
        claim["matched_fragments"] = matched_fragments
        claim["unmatched_fragments"] = unmatched_fragments
        claim["matched_pages"] = sorted(matched_pages)

        claim["evidence_exists"] = len(matched_fragments) > 0

        claim["page_match"] = (
            set(claim.get("page_number") or []).issubset(matched_pages)
            if claim.get("page_number")
            else False
        )

        if not fragments:
            claim["validation_status"] = "NO USABLE EVIDENCE"

        elif len(matched_fragments) == len(fragments):
            claim["validation_status"] = "ALL FRAGMENTS MATCH"

        elif matched_fragments:
            claim["validation_status"] = "PARTIAL EVIDENCE MATCH"

        else:
            claim["validation_status"] = "NO EVIDENCE MATCH"

    return report

In [ ]:
validated_report_v2 = validate_evidence_v2(
    verification_report_v2,
    paper_text
)

for claim in validated_report_v2["claims"]:

    print("\n" + "=" * 60)
    print("CLAIM:", claim["claim"])
    print("GEMINI STATUS:", claim["status"])
    print("VALIDATION:", claim["validation_status"])
    print("MATCHED PAGES:", claim["matched_pages"])
    print("MATCHED FRAGMENTS:", len(claim["matched_fragments"]))
    print("UNMATCHED FRAGMENTS:", len(claim["unmatched_fragments"]))


CLAIM: The research paper is a survey article providing an analysis of deepfake video generation and detection technologies.
GEMINI STATUS: SUPPORTED
VALIDATION: ALL FRAGMENTS MATCH
MATCHED PAGES: [1]
MATCHED FRAGMENTS: 1
UNMATCHED FRAGMENTS: 0

CLAIM: The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.
GEMINI STATUS: SUPPORTED
VALIDATION: NO EVIDENCE MATCH
MATCHED PAGES: []
MATCHED FRAGMENTS: 0
UNMATCHED FRAGMENTS: 1

CLAIM: The study adopts a survey-based review approach and identifies data, training, and reliability as the main categories of challenges.
GEMINI STATUS: SUPPORTED
VALIDATION: ALL FRAGMENTS MATCH
MATCHED PAGES: [1]
MATCHED FRAGMENTS: 1
UNMATCHED FRAGMENTS: 0

CLAIM: Deep learning-based methods currently dominate deepfake detection, with GANs serving as a key tool for creation and a subject for detection research.
GEMINI STATUS: SUPPORTED
VALIDATION: PARTIAL EVIDENCE MATCH
MATCHED PAGES: [1]
MA

In [ ]:
async def semantic_evidence_check(report, paper_text):

    # Extract individual PDF pages
    pages = {}
    current_page = None

    for line in paper_text.splitlines():

        match = re.match(r"--- PAGE (\d+) ---", line)

        if match:
            current_page = int(match.group(1))
            pages[current_page] = ""

        elif current_page is not None:
            pages[current_page] += line + "\n"

    results = []

    for index, claim in enumerate(report["claims"], start=1):

        # Retrieve only the pages where evidence matched
        evidence_pages = claim.get("matched_pages", [])

        source_evidence = ""

        for page_number in evidence_pages:
            source_evidence += (
                f"\nPAGE {page_number}:\n"
                f"{pages.get(page_number, '')}\n"
            )

        if not source_evidence:
            source_evidence = "No matching source evidence was found."

        prompt = f"""
You are an academic evidence evaluation assistant.

Evaluate whether the original paper evidence actually supports
the given claim.

CLAIM:
{claim['claim']}

GEMINI'S PREVIOUS STATUS:
{claim['status']}

QUOTATION PROVIDED:
{claim.get('evidence', '')}

ORIGINAL SOURCE PAGES:
{source_evidence}

Evaluate these points:

1. Does the source directly support the claim?
2. Which parts of the claim are supported?
3. Which parts are unsupported?
4. Does the claim make an inference beyond the source?
5. What would be a more accurate version of the claim?

Rules:
- Use only the supplied source pages.
- Do not assume that a matching quotation proves a claim.
- Do not confuse reviewed datasets with experimentally used datasets.
- Do not confuse literature findings with original experiments.
- If the source pages are insufficient, say so.
- Never invent evidence.

Return only valid JSON:

{{
  "claim_number": {index},
  "semantic_status": "SUPPORTED / PARTIALLY SUPPORTED / NOT SUPPORTED / INSUFFICIENT EVIDENCE",
  "supported_parts": [],
  "unsupported_parts": [],
  "reasoning": "",
  "suggested_correction": ""
}}
"""

        response = gemini_client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config={"response_mime_type": "application/json"}
        )

        results.append(json.loads(response.text))

        print(f"Checked claim {index}/{len(report['claims'])}")

    return results

In [ ]:
semantic_results = await semantic_evidence_check(
    validated_report_v2,
    paper_text
)

for result in semantic_results:

    print("\n" + "=" * 60)
    print("CLAIM:", result["claim_number"])
    print("STATUS:", result["semantic_status"])
    print("SUPPORTED PARTS:", result["supported_parts"])
    print("UNSUPPORTED PARTS:", result["unsupported_parts"])
    print("REASONING:", result["reasoning"])
    print("SUGGESTED CORRECTION:", result["suggested_correction"])

Checked claim 1/7


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
import asyncio
import json

async def semantic_evidence_check(report, paper_text):

    pages = {}
    current_page = None

    for line in paper_text.splitlines():

        match = re.match(r"--- PAGE (\d+) ---", line)

        if match:
            current_page = int(match.group(1))
            pages[current_page] = ""

        elif current_page is not None:
            pages[current_page] += line + "\n"

    # Keep results outside the loop
    results = []

    for index, claim in enumerate(report["claims"], start=1):

        evidence_pages = claim.get("matched_pages", [])
        source_evidence = ""

        for page_number in evidence_pages:
            source_evidence += (
                f"\nPAGE {page_number}:\n"
                f"{pages.get(page_number, '')}\n"
            )

        if not source_evidence:
            source_evidence = "No matching source evidence was found."

        prompt = f"""
You are an academic evidence evaluation assistant.

CLAIM:
{claim['claim']}

PREVIOUS STATUS:
{claim['status']}

QUOTATION:
{claim.get('evidence', '')}

ORIGINAL SOURCE PAGES:
{source_evidence}

Evaluate:
1. Does the evidence directly support the claim?
2. Which parts are supported?
3. Which parts are unsupported?
4. Does the claim make an inference beyond the evidence?
5. What is a more accurate version?

Use only the supplied source pages.
Do not invent evidence.

Return only valid JSON:

{{
  "claim_number": {index},
  "semantic_status": "SUPPORTED / PARTIALLY SUPPORTED / NOT SUPPORTED / INSUFFICIENT EVIDENCE",
  "supported_parts": [],
  "unsupported_parts": [],
  "reasoning": "",
  "suggested_correction": ""
}}
"""

        # Retry temporary API failures
        for attempt in range(3):

            try:
                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config={"response_mime_type": "application/json"}
                )

                result = json.loads(response.text)
                results.append(result)

                print(f"Claim {index}/7 completed successfully.")
                break

            except Exception as error:

                print(f"Claim {index}: Attempt {attempt + 1} failed.")
                print("Error:", str(error)[:250])

                if attempt < 2:
                    wait_time = 10 * (attempt + 1)
                    print(f"Retrying in {wait_time} seconds...")
                    await asyncio.sleep(wait_time)

                else:
                    results.append({
                        "claim_number": index,
                        "semantic_status": "API_ERROR",
                        "supported_parts": [],
                        "unsupported_parts": [],
                        "reasoning": str(error),
                        "suggested_correction": ""
                    })

    return results

In [ ]:
semantic_results = await semantic_evidence_check(
    validated_report_v2,
    paper_text
)

Claim 1/7 completed successfully.
Claim 2/7 completed successfully.
Claim 3/7 completed successfully.
Claim 4/7 completed successfully.
Claim 5/7 completed successfully.
Claim 6/7 completed successfully.
Claim 7/7 completed successfully.


In [ ]:
for result in semantic_results:

    print("\n" + "=" * 65)
    print("CLAIM:", result["claim_number"])
    print("STATUS:", result["semantic_status"])

    print("\nSupported parts:")
    for item in result["supported_parts"]:
        print("-", item)

    print("\nUnsupported parts:")
    for item in result["unsupported_parts"]:
        print("-", item)

    print("\nReasoning:")
    print(result["reasoning"])

    print("\nSuggested correction:")
    print(result["suggested_correction"])


CLAIM: 1
STATUS: SUPPORTED

Supported parts:
- The research paper is a survey article.
- The paper provides an analysis of deepfake video generation and detection technologies.

Unsupported parts:

Reasoning:
The abstract of the provided paper explicitly states, 'This survey paper aims to thoroughly analyse deepfake video generation and detection.' This directly confirms both the nature of the paper as a survey and its core analytical focus.

Suggested correction:
None required.

CLAIM: 2
STATUS: INSUFFICIENT EVIDENCE

Supported parts:

Unsupported parts:
- The research objective is to analyze deepfake video generation and detection, categorize challenges, and identify research gaps.

Reasoning:
The provided source information explicitly states that 'No matching source evidence was found' regarding the original source pages. Therefore, the claim cannot be verified using the provided materials.

Suggested correction:
This claim cannot be evaluated as true or false because no source evi

In [ ]:
await research_assistant(
    filename="Deepfake.pdf",
    task="summarize"
)

NameError: name 'research_assistant' is not defined

In [ ]:
async def research_assistant(filename=None, task="list", query=None):

    # Connect to the MCP server
    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            # TASK 1: List uploaded research papers
            if task == "list":

                result = await session.call_tool(
                    "list_documents",
                    arguments={}
                )

                print("Available research papers:\n")

                for item in result.content:
                    if item.type == "text":
                        print(item.text)

                return

            # TASK 2: Search across research papers
            elif task == "search":

                if not query:
                    return "Please provide a search query."

                result = await session.call_tool(
                    "search_documents",
                    arguments={"query": query}
                )

                return "\n".join(
                    item.text
                    for item in result.content
                    if item.type == "text"
                )

            # TASK 3: Summarise a research paper
            elif task == "summarize":

                if not filename:
                    return "Please provide a PDF filename."

                result = await session.call_tool(
                    "read_document",
                    arguments={"filename": filename}
                )

                paper_text = "\n".join(
                    item.text
                    for item in result.content
                    if item.type == "text"
                )

                if not paper_text.strip():
                    return "No readable text found in the PDF."

                # Split the paper into manageable chunks
                chunk_size = 12000
                overlap = 500
                chunks = []
                start = 0

                while start < len(paper_text):
                    end = start + chunk_size
                    chunks.append(paper_text[start:end])
                    start += chunk_size - overlap

                print("Paper:", filename)
                print("Total chunks:", len(chunks))

                # MAP: Summarise each chunk
                chunk_summaries = []

                for i, chunk in enumerate(chunks):

                    print(f"Processing chunk {i+1}/{len(chunks)}")

                    prompt = f"""
                    You are an academic research assistant.

                    Summarise this section of the research paper.

                    Preserve important technical details.
                    Do not invent information.
                    Distinguish survey findings from original experiments.

                    SECTION:
                    {chunk}
                    """

                    response = gemini_client.models.generate_content(
                        model=MODEL_NAME,
                        contents=prompt
                    )

                    chunk_summaries.append(response.text)

                # REDUCE: Combine all chunk summaries
                print("\nCombining summaries...")

                final_prompt = f"""
                You are an academic research assistant.

                Create a comprehensive summary of this research paper.

                Use these headings:

                1. Research Objective
                2. Methodology
                3. Models and Algorithms
                4. Dataset Analysis
                5. Experimental Results
                6. Limitations
                7. Future Research Directions

                Important:
                - Use only the supplied section summaries.
                - Do not invent datasets or numerical results.
                - Distinguish datasets reviewed from datasets used
                  in original experiments.
                - Write Not Reported when information is unavailable.
                - Use simple English.

                SECTION SUMMARIES:

                {"".join(chunk_summaries)}
                """

                final_response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=final_prompt
                )

                return final_response.text

            else:
                return (
                    "Unsupported task. Available tasks: "
                    "list, search, summarize"
                )

In [ ]:
summary_result = await research_assistant(
    filename="Deepfake.pdf",
    task="summarize"
)

print(summary_result)

Paper: Deepfake.pdf
Total chunks: 13
Processing chunk 1/13
Processing chunk 2/13
Processing chunk 3/13
Processing chunk 4/13
Processing chunk 5/13
Processing chunk 6/13
Processing chunk 7/13
Processing chunk 8/13
Processing chunk 9/13
Processing chunk 10/13
Processing chunk 11/13
Processing chunk 12/13
Processing chunk 13/13

Combining summaries...
This research summary is based on the paper *“Deepfake video detection: challenges and opportunities”* (Kaur et al., 2024).

### 1. Research Objective
The objective of this research was to provide a systematic review of the deepfake ecosystem. The authors aimed to classify deepfake manipulation techniques, evaluate existing detection methodologies, and identify critical gaps in current academic research to establish a framework for more robust, efficient, and real-time detection solutions.

### 2. Methodology
The authors conducted a Systematic Literature Review (SLR) by querying major academic databases including IEEE Xplore, ACM Digital Lib

In [ ]:
verification_report_v3 = await verify_summary_v2(
    "Deepfake.pdf",
    summary_result
)

print(json.dumps(
    verification_report_v3,
    indent=2,
    ensure_ascii=False
))

{
  "paper_name": "Artificial Intelligence Review (2024) 57:159",
  "claims": [
    {
      "claim": "The research objective was to provide a systematic review of the deepfake ecosystem, classifying manipulation techniques, evaluating detection methodologies, and identifying gaps for real-time solutions.",
      "status": "SUPPORTED",
      "evidence": "This survey paper aims to thoroughly analyse deepfake video generation and detection. The paper’s main contribution is the classification of the many challenges encountered while detecting deepfake videos. . . This entails developing robust models for real-time detection.",
      "page_number": [
        1
      ],
      "evidence_quality": "STRONG",
      "explanation": "The abstract explicitly states the goal of analyzing generation and detection, the classification of challenges, and the need for real-time detection as a future direction.",
      "confidence": "HIGH"
    },
    {
      "claim": "The authors conducted an SLR, selectin

In [ ]:
import json
import asyncio
import re

async def extract_claims(summary):

    prompt = f"""
You are an academic research fact-checking assistant.

Extract EVERY factual claim from the research summary below.

SUMMARY:
{summary}

Instructions:

1. Extract all factual claims from every section.
2. Do not skip technical details.
3. Separate compound sentences into atomic claims.
4. Include numerical values, dataset names, model names,
   algorithms, metrics, methodologies and future directions.
5. Do not create new claims that are absent from the summary.
6. Ignore headings and general introductory statements.
7. Preserve the original meaning.

Return only valid JSON in this format:

{{
  "claims": [
    {{
      "claim_id": 1,
      "category": "Methodology",
      "claim": "The authors reviewed 132 research papers."
    }}
  ]
}}
"""

    for attempt in range(3):
        try:
            response = gemini_client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={
                    "response_mime_type": "application/json"
                }
            )

            data = json.loads(response.text)

            # Correct numbering ourselves
            for index, claim in enumerate(data["claims"], start=1):
                claim["claim_id"] = index

            return data["claims"]

        except Exception as error:
            print(f"Attempt {attempt + 1} failed:", str(error)[:200])

            if attempt < 2:
                await asyncio.sleep(10 * (attempt + 1))

    raise RuntimeError("Claim extraction failed after three attempts.")

In [ ]:
extracted_claims = await extract_claims(summary_result)

print("Total factual claims:", len(extracted_claims))

for claim in extracted_claims:
    print(
        f"\nClaim {claim['claim_id']}"
        f"\nCategory: {claim['category']}"
        f"\nStatement: {claim['claim']}"
    )

Total factual claims: 36

Claim 1
Category: Source
Statement: The research summary is based on the paper 'Deepfake video detection: challenges and opportunities' by Kaur et al. (2024).

Claim 2
Category: Objective
Statement: The research objective was to provide a systematic review of the deepfake ecosystem.

Claim 3
Category: Objective
Statement: The authors aimed to classify deepfake manipulation techniques.

Claim 4
Category: Objective
Statement: The authors aimed to evaluate existing detection methodologies.

Claim 5
Category: Objective
Statement: The authors aimed to identify critical gaps in current academic research.

Claim 6
Category: Methodology
Statement: The authors conducted a Systematic Literature Review (SLR).

Claim 7
Category: Methodology
Statement: The authors queried IEEE Xplore, ACM Digital Library, Scopus, and arXiv.

Claim 8
Category: Methodology
Statement: The search criteria included keywords such as 'deepfake,' 'video manipulation,' and 'deep learning'.

Claim 9

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

def retrieve_relevant_pages(claims, paper_text, top_k=3):

    # 1. Extract individual pages
    pages = {}
    current_page = None

    for line in paper_text.splitlines():

        match = re.match(r"--- PAGE (\d+) ---", line)

        if match:
            current_page = int(match.group(1))
            pages[current_page] = ""

        elif current_page is not None:
            pages[current_page] += line + "\n"

    page_numbers = list(pages.keys())
    page_texts = [pages[number] for number in page_numbers]

    # 2. Convert PDF pages into TF-IDF vectors
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=30000,
        ngram_range=(1, 2)
    )

    page_vectors = vectorizer.fit_transform(page_texts)

    # 3. Retrieve relevant pages for every claim
    retrieved_results = []

    for claim in claims:

        query = claim["claim"]

        query_vector = vectorizer.transform([query])

        scores = cosine_similarity(
            query_vector,
            page_vectors
        ).flatten()

        top_indices = scores.argsort()[::-1][:top_k]

        relevant_pages = []

        for index in top_indices:

            if scores[index] <= 0:
                continue

            page_number = page_numbers[index]

            relevant_pages.append({
                "page_number": page_number,
                "similarity_score": round(
                    float(scores[index]), 4
                ),
                "text": pages[page_number]
            })

        retrieved_results.append({
            "claim_id": claim["claim_id"],
            "claim": query,
            "category": claim["category"],
            "relevant_pages": relevant_pages
        })

    return retrieved_results

In [ ]:
retrieved_claims = retrieve_relevant_pages(
    extracted_claims,
    paper_text,
    top_k=3
)

print("Total claims processed:", len(retrieved_claims))

for item in retrieved_claims:

    print("\n" + "=" * 65)
    print("CLAIM:", item["claim_id"])
    print("CATEGORY:", item["category"])
    print("STATEMENT:", item["claim"])

    print("\nRelevant pages:")

    for page in item["relevant_pages"]:

        print(
            f"Page {page['page_number']} | "
            f"Similarity: {page['similarity_score']}"
        )

Total claims processed: 36

CLAIM: 1
CATEGORY: Source
STATEMENT: The research summary is based on the paper 'Deepfake video detection: challenges and opportunities' by Kaur et al. (2024).

Relevant pages:
Page 8 | Similarity: 0.2167
Page 7 | Similarity: 0.1793
Page 1 | Similarity: 0.178

CLAIM: 2
CATEGORY: Objective
STATEMENT: The research objective was to provide a systematic review of the deepfake ecosystem.

Relevant pages:
Page 8 | Similarity: 0.1815
Page 9 | Similarity: 0.0693
Page 41 | Similarity: 0.0606

CLAIM: 3
CATEGORY: Objective
STATEMENT: The authors aimed to classify deepfake manipulation techniques.

Relevant pages:
Page 5 | Similarity: 0.0789
Page 36 | Similarity: 0.0726
Page 32 | Similarity: 0.0431

CLAIM: 4
CATEGORY: Objective
STATEMENT: The authors aimed to evaluate existing detection methodologies.

Relevant pages:
Page 5 | Similarity: 0.0889
Page 38 | Similarity: 0.0847
Page 7 | Similarity: 0.0609

CLAIM: 5
CATEGORY: Objective
STATEMENT: The authors aimed to identif

In [ ]:
import json
import re
import asyncio

# Store results so completed claims aren't lost if a later call fails
claim_verification_results = []

async def verify_all_claims(retrieved_claims):

    completed_ids = {
        item["claim_id"]
        for item in claim_verification_results
    }

    total = len(retrieved_claims)

    for item in retrieved_claims:

        claim_id = item["claim_id"]

        # Skip claims already processed
        if claim_id in completed_ids:
            continue

        # Prepare source pages for Gemini
        source_pages = ""

        for page in item["relevant_pages"]:

            source_pages += f"""
            ORIGINAL PDF PAGE: {page['page_number']}
            TF-IDF SIMILARITY: {page['similarity_score']}

            PAGE CONTENT:
            {page['text'][:8000]}

            --------------------------------
            """

        if not source_pages:
            claim_verification_results.append({
                "claim_id": claim_id,
                "claim": item["claim"],
                "status": "INSUFFICIENT EVIDENCE",
                "evidence": "",
                "page_number": [],
                "explanation": "No relevant source pages retrieved.",
                "suggested_correction": ""
            })

            print(f"Claim {claim_id}/{total}: No source pages.")
            continue

        prompt = f"""
You are an academic research fact-checking assistant.

Verify the following factual claim against the original
research paper pages supplied below.

CLAIM ID:
{claim_id}

CATEGORY:
{item['category']}

CLAIM:
{item['claim']}

ORIGINAL PDF SOURCE PAGES:
{source_pages}

VERIFICATION RULES:

1. Evaluate the complete claim, not just individual keywords.
2. Use only the supplied original PDF pages.
3. Provide exact quotations from the paper.
4. Give the original PDF page number for each quotation.
5. Separate supported and unsupported parts.
6. Do not invent quotations, results, or page numbers.
7. Do not treat TF-IDF similarity as proof of correctness.
8. If the supplied pages are insufficient, use
   INSUFFICIENT EVIDENCE.
9. Use CONTRADICTED only when the source explicitly
   contradicts the claim.
10. Do not confuse reported literature results with
    experiments performed by the survey authors.
11. Do not confuse datasets reviewed with datasets
    used in original experiments.

Return only valid JSON:

{{
  "claim_id": {claim_id},
  "status": "SUPPORTED / PARTIALLY SUPPORTED / NOT SUPPORTED / CONTRADICTED / INSUFFICIENT EVIDENCE",
  "supported_parts": [],
  "unsupported_parts": [],
  "evidence": [
    {{
      "quotation": "Exact quotation from the paper",
      "page_number": 1
    }}
  ],
  "explanation": "",
  "suggested_correction": ""
}}
"""

        # Retry temporary API failures
        for attempt in range(3):

            try:

                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config={
                        "response_mime_type": "application/json"
                    }
                )

                raw = response.text.strip()

                raw = re.sub(
                    r"^```(?:json)?\s*|\s*```$",
                    "",
                    raw
                )

                result = json.loads(raw)

                # Trust our Python claim ID, not model-generated numbering
                result["claim_id"] = claim_id
                result["claim"] = item["claim"]

                claim_verification_results.append(result)

                print(
                    f"Claim {claim_id}/{total} "
                    f"completed: {result['status']}"
                )

                break

            except Exception as error:

                print(
                    f"Claim {claim_id}: "
                    f"Attempt {attempt + 1} failed: "
                    f"{str(error)[:150]}"
                )

                if attempt < 2:
                    await asyncio.sleep(10 * (attempt + 1))

                else:
                    claim_verification_results.append({
                        "claim_id": claim_id,
                        "claim": item["claim"],
                        "status": "API_ERROR",
                        "evidence": [],
                        "explanation": str(error),
                        "suggested_correction": ""
                    })

    return claim_verification_results

In [ ]:
final_claim_report = await verify_all_claims(
    retrieved_claims
)

print("\nTotal processed:", len(final_claim_report))

Claim 1/36 completed: SUPPORTED
Claim 2/36 completed: NOT SUPPORTED
Claim 3/36 completed: NOT SUPPORTED
Claim 4/36 completed: SUPPORTED
Claim 5/36 completed: SUPPORTED
Claim 6/36 completed: SUPPORTED
Claim 7/36 completed: PARTIALLY SUPPORTED
Claim 8/36 completed: SUPPORTED
Claim 9/36 completed: SUPPORTED
Claim 10/36 completed: PARTIALLY SUPPORTED
Claim 11/36 completed: SUPPORTED
Claim 12/36 completed: SUPPORTED
Claim 13/36 completed: SUPPORTED
Claim 14/36 completed: SUPPORTED
Claim 15/36 completed: SUPPORTED
Claim 16/36 completed: SUPPORTED
Claim 17/36 completed: SUPPORTED
Claim 18/36 completed: INSUFFICIENT EVIDENCE
Claim 19/36 completed: SUPPORTED
Claim 20/36 completed: SUPPORTED
Claim 21/36 completed: SUPPORTED
Claim 22/36 completed: SUPPORTED
Claim 23/36 completed: SUPPORTED
Claim 24/36 completed: SUPPORTED
Claim 25/36 completed: SUPPORTED
Claim 26/36 completed: SUPPORTED
Claim 27/36 completed: SUPPORTED
Claim 28/36 completed: PARTIALLY SUPPORTED
Claim 29/36 completed: SUPPORTED
Cl

In [ ]:
for result in final_claim_report:

    print("\n" + "=" * 65)
    print("CLAIM ID:", result["claim_id"])
    print("CLAIM:", result["claim"])
    print("STATUS:", result["status"])

    print("\nEVIDENCE:")

    for evidence in result.get("evidence", []):
        print("Quotation:", evidence.get("quotation"))
        print("Page:", evidence.get("page_number"))

    print("\nSUPPORTED PARTS:")
    for part in result.get("supported_parts", []):
        print("-", part)

    print("\nUNSUPPORTED PARTS:")
    for part in result.get("unsupported_parts", []):
        print("-", part)

    print("\nEXPLANATION:")
    print(result.get("explanation", ""))

    print("\nSUGGESTED CORRECTION:")
    print(result.get("suggested_correction", ""))


CLAIM ID: 1
CLAIM: The research summary is based on the paper 'Deepfake video detection: challenges and opportunities' by Kaur et al. (2024).
STATUS: SUPPORTED

EVIDENCE:
Quotation: Artificial Intelligence Review (2024) 57:159... Deepfake video detection: challenges and opportunities... Achhardeep Kaur1 · Azadeh Noori Hoshyar2 · Vidya Saikrishna3 · Selena Firmin1 · Feng Xia4
Page: 1

SUPPORTED PARTS:
- The research summary is based on the paper 'Deepfake video detection: challenges and opportunities' by Kaur et al. (2024).

UNSUPPORTED PARTS:

EXPLANATION:
The provided source document clearly lists the title as 'Deepfake video detection: challenges and opportunities' and lists 'Achhardeep Kaur' as the first author with a publication date of 2024.

SUGGESTED CORRECTION:


CLAIM ID: 2
CLAIM: The research objective was to provide a systematic review of the deepfake ecosystem.
STATUS: NOT SUPPORTED

EVIDENCE:
Quotation: The main objective of our systematic literature review (SLR) is to ex

In [ ]:
import json
import re
import time

def extract_json(text):
    """Extract JSON even if Gemini adds markdown code fences."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


async def verify_claims_in_batches(retrieved_claims, batch_size=9):

    all_results = []

    total_batches = (
        len(retrieved_claims) + batch_size - 1
    ) // batch_size

    for batch_number, start in enumerate(
        range(0, len(retrieved_claims), batch_size), start=1
    ):

        batch = retrieved_claims[start:start + batch_size]

        print(f"\nProcessing batch {batch_number}/{total_batches}")

        # Collect relevant pages without sending duplicate pages.
        page_collection = {}

        for item in batch:
            for page in item["relevant_pages"][:2]:
                page_number = page["page_number"]

                if page_number not in page_collection:
                    page_collection[page_number] = page["text"][:3500]

        claims_for_prompt = [
            {
                "claim_id": item["claim_id"],
                "category": item["category"],
                "claim": item["claim"]
            }
            for item in batch
        ]

        prompt = f"""
You are an academic research paper fact-checking assistant.

Your task is to verify each claim using ONLY the supplied paper pages.

CLAIMS:
{json.dumps(claims_for_prompt, indent=2)}

SOURCE PAGES:
{json.dumps(page_collection, indent=2)}

For every claim, return:

1. claim_id
2. status — choose exactly one:
   - SUPPORTED
   - PARTIALLY SUPPORTED
   - CONTRADICTED
   - INSUFFICIENT EVIDENCE

3. supported_parts
4. unsupported_parts
5. evidence — exact quotations from the supplied pages.
   Each quotation must include its page_number.
6. explanation — brief reasoning.
7. suggested_correction — provide a corrected claim if necessary.

IMPORTANT RULES:
- Do not invent quotations, page numbers, datasets, or results.
- A similar topic is not sufficient evidence.
- If the supplied pages do not establish a claim, use
  INSUFFICIENT EVIDENCE.
- Distinguish a paper's own experiments from results
  reported by studies it reviews.
- Numerical claims require evidence for the exact numbers.
- Return every claim exactly once.
- Return valid JSON only, using this structure:

{{
  "results": [
    {{
      "claim_id": 1,
      "status": "SUPPORTED",
      "supported_parts": [],
      "unsupported_parts": [],
      "evidence": [
        {{
          "quotation": "Exact text from the paper",
          "page_number": 1
        }}
      ],
      "explanation": "Reason for the decision.",
      "suggested_correction": ""
    }}
  ]
}}
"""

        # Retry if Gemini temporarily fails.
        for attempt in range(3):
            try:
                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt
                )

                batch_results = extract_json(response.text)["results"]

                # Use the original claim IDs, not IDs invented by the model.
                expected_ids = {item["claim_id"] for item in batch}
                returned_ids = {
                    item["claim_id"] for item in batch_results
                }

                if returned_ids != expected_ids:
                    raise ValueError(
                        "Gemini returned missing or unexpected claim IDs."
                    )

                all_results.extend(batch_results)

                print(
                    f"Batch {batch_number} completed: "
                    f"{len(batch_results)} claims verified."
                )

                break

            except Exception as error:
                print(f"Attempt {attempt + 1} failed: {error}")

                if attempt == 2:
                    print("Batch could not be completed.")
                    raise

                time.sleep(10 * (attempt + 1))

    return all_results


# Run verification
batch_verification_results = await verify_claims_in_batches(
    retrieved_claims,
    batch_size=9
)

print("\nVerification completed!")
print("Total claims checked:", len(batch_verification_results))


Processing batch 1/4
Batch 1 completed: 9 claims verified.

Processing batch 2/4
Batch 2 completed: 9 claims verified.

Processing batch 3/4
Batch 3 completed: 9 claims verified.

Processing batch 4/4
Batch 4 completed: 9 claims verified.

Verification completed!
Total claims checked: 36


In [ ]:
for result in batch_verification_results:
    print("\nClaim ID:", result["claim_id"])
    print("Status:", result["status"])
    print("Explanation:", result["explanation"])
    print("Suggested correction:", result["suggested_correction"])


Claim ID: 1
Status: SUPPORTED
Explanation: The paper title, authors, and year (2024) are confirmed on page 1.
Suggested correction: 

Claim ID: 2
Status: SUPPORTED
Explanation: The authors explicitly state the objective of their systematic literature review.
Suggested correction: 

Claim ID: 3
Status: SUPPORTED
Explanation: The authors state the objective to explore the evolution of generation techniques and categorize detection challenges.
Suggested correction: 

Claim ID: 4
Status: SUPPORTED
Explanation: The authors explicitly list the assessment of detection methods as a sub-objective.
Suggested correction: 

Claim ID: 5
Status: SUPPORTED
Explanation: The authors clearly state the intent to identify challenges and research gaps.
Suggested correction: 

Claim ID: 6
Status: SUPPORTED
Explanation: The authors identify their methodology as an SLR on page 8.
Suggested correction: 

Claim ID: 7
Status: SUPPORTED
Explanation: The paper confirms searching these specific databases among oth

In [ ]:
# Generate summaries of both research papers

paper1_summary = await research_assistant(
    filename="Brain_tumor2.pdf",
    task="summarize"
)

paper2_summary = await research_assistant(
    filename="Brain_tuomr1.pdf",
    task="summarize"
)

print("Both paper summaries generated successfully!")

Paper: Brain_tumor2.pdf
Total chunks: 6
Processing chunk 1/6
Processing chunk 2/6
Processing chunk 3/6
Processing chunk 4/6
Processing chunk 5/6
Processing chunk 6/6

Combining summaries...
Paper: Brain_tuomr1.pdf
Total chunks: 10
Processing chunk 1/10
Processing chunk 2/10
Processing chunk 3/10
Processing chunk 4/10
Processing chunk 5/10
Processing chunk 6/10
Processing chunk 7/10
Processing chunk 8/10
Processing chunk 9/10
Processing chunk 10/10

Combining summaries...
Both paper summaries generated successfully!


In [ ]:
comparison_prompt = f"""
You are an academic research comparison assistant.

Compare the following two research papers.

PAPER 1:
{paper1_summary}

PAPER 2:
{paper2_summary}

Compare them using these headings:

1. Research Objective
2. Methodology
3. Models and Algorithms
4. Dataset
5. Evaluation Metrics
6. Experimental Results
7. Strengths
8. Limitations
9. Key Differences
10. Overall Comparison

Rules:
- Use only information provided in the summaries.
- Do not invent datasets, models, or performance metrics.
- If information is missing, write "Not reported".
- Distinguish reported experimental results from survey findings.
- Do not declare one paper better without sufficient evidence.
- Use simple English.
"""

comparison_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents=comparison_prompt
)

print(comparison_response.text)

This comparison analyzes two research papers focused on automated brain tumor detection using deep learning.

### 1. Research Objective
Both papers aim to develop high-precision, automated systems for detecting and classifying three types of brain tumors: meningioma, glioma, and pituitary. Both seek to replace subjective and time-consuming manual radiological diagnosis.

### 2. Methodology
*   **Paper 1:** Uses a segmentation-free pipeline, preprocessing via NumPy array conversion, and standard data augmentation (flipping, rotating, etc.). It uses an Adam optimizer and dropout layers to manage overfitting.
*   **Paper 2:** Uses a three-stage preprocessing strategy (grayscale, resizing, noise reduction). It employs the *Albumentations* library for geometric augmentation and utilizes transfer learning from the COCO dataset.

### 3. Models and Algorithms
*   **Paper 1:** Developed a custom 23-layer CNN and a hybrid model combining this CNN with pre-trained VGG16.
*   **Paper 2:** Uses a m

In [ ]:
gap_prompt = f"""
You are an academic research analyst.

Analyse the following two brain tumour detection research papers
and identify potential research gaps.

PAPER 1:
{paper1_summary}

PAPER 2:
{paper2_summary}

Perform the following analysis:

1. Common Limitations
   Identify limitations shared by both papers.

2. Paper-Specific Limitations
   Identify limitations unique to each paper.

3. Unresolved Research Problems
   Identify problems that neither paper adequately addresses.

4. Potential Research Gaps
   Suggest 5 meaningful research gaps based on the papers.

5. Proposed Research Directions
   For each gap, suggest a possible approach to investigate it.

6. Potential Project Topics
   Suggest 3 practical research project titles.

IMPORTANT RULES:
- Use only information provided in the summaries.
- Clearly distinguish explicitly reported limitations
  from your own inferred research gaps.
- Do not present an inferred gap as a confirmed fact.
- Do not invent experimental results.
- Consider differences in datasets, architectures,
  evaluation metrics and clinical validation.
- Use simple academic English.

For each research gap, include:
- Gap
- Supporting observations from the papers
- Why it matters
- Possible research direction
- Evidence type: Explicit or Inferred
"""

gap_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents=gap_prompt
)

print(gap_response.text)

This analysis evaluates the provided research papers on brain tumor detection, focusing on identifying gaps and directions for future inquiry.

### 1. Common Limitations
*   **Lack of Clinical Validation:** Neither model has been validated against real-world, clinical-grade patient data; both rely exclusively on public/academic datasets.
*   **Dependency on Large/Labeled Data:** Both papers acknowledge that high-performance models require massive amounts of professionally annotated data, which is labor-intensive and expensive.
*   **Overfitting Sensitivity:** Both studies highlight the risk of model failure or overfitting when dealing with datasets that lack sufficient diversity or size.

### 2. Paper-Specific Limitations
*   **Paper 1:** 
    *   **Architecture Sensitivity:** The 23-layer CNN is specifically susceptible to poor performance (under 55% accuracy) on small datasets without hybrid integration.
*   **Paper 2:**
    *   **Morphological Challenges:** The model struggles to de

In [ ]:
async def ask_research_question(question):

    if not question.strip():
        return "Please enter a question."

    # Step 1: Retrieve relevant information through MCP
    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "search_documents",
                arguments={"query": question}
            )

            retrieved_context = "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )

    if not retrieved_context.strip():
        return "No relevant information found in the research papers."

    # Step 2: Ask Gemini to answer using retrieved information
    prompt = f"""
You are an academic research assistant.

Answer the user's question using the research paper
information provided below.

USER QUESTION:
{question}

RETRIEVED RESEARCH INFORMATION:
{retrieved_context}

INSTRUCTIONS:
1. Use only the retrieved research information.
2. Do not invent facts, models, datasets or results.
3. Mention the PDF filename when using information from a paper.
4. If the information is insufficient, clearly say so.
5. If papers provide different findings, explain the differences.
6. Use simple and clear English.
7. Organise the answer using headings or bullet points when useful.

ANSWER:
"""

    response = gemini_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text

In [ ]:
answer = await ask_research_question(
    "Compare the deep learning architectures "
    "used in the brain tumour research papers."
)

print(answer)

Based on the provided research papers, here is a comparison of the deep learning architectures used for brain tumor research:

### **Brain_tuomr1.pdf**
*   **Architecture Used:** This study utilized the **YOLOv7** model.
*   **Methodology:** The researchers focused on fine-tuning the state-of-the-art YOLOv7 model through transfer learning to identify and locate gliomas, meningioma, and pituitary brain tumors in MRI scans.
*   **Outcome:** The model achieved 99.5% accuracy in their analysis.

### **Brain_tumor2.pdf**
*   **Architecture Used:** This study employed **Deep Convolutional Neural Networks (CNNs)**.
*   **Methodology:** The paper proposed two deep learning models based on CNN architectures to classify both binary (normal vs. abnormal) and multiclass (meningioma, glioma, and pituitary) brain tumors. The authors noted that these methods are advantageous because they do not require the extraction of handcrafted features, unlike traditional machine learning methods.
*   **Outcome:

In [ ]:
# Install Streamlit
!pip install -q streamlit

import os
from google.colab import userdata

# Set Gemini API key securely
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Project paths
PROJECT_DIR = "/content/mcp_research_assistant"
DOCUMENTS_DIR = os.path.join(PROJECT_DIR, "documents")
SERVER_PATH = os.path.join(PROJECT_DIR, "server.py")

# Check project files
print("Streamlit installation: Ready")
print("Project directory:", os.path.exists(PROJECT_DIR))
print("MCP server:", os.path.exists(SERVER_PATH))
print("Documents directory:", os.path.exists(DOCUMENTS_DIR))
print("PDF files:", os.listdir(DOCUMENTS_DIR))

Streamlit installation: Ready
Project directory: True
MCP server: True
Documents directory: True
PDF files: ['Deepfake.pdf', 'house_prediction.pdf', 'llm.pdf', 'Brain_tumor2.pdf', 'Brain_tuomr1.pdf']


In [ ]:
%%writefile /content/mcp_research_assistant/app.py

import os
import sys
import asyncio
import streamlit as st

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


# --------------------------------
# 1. PROJECT CONFIGURATION
# --------------------------------

PROJECT_DIR = "/content/mcp_research_assistant"
DOCUMENTS_DIR = os.path.join(PROJECT_DIR, "documents")
SERVER_PATH = os.path.join(PROJECT_DIR, "server.py")

MODEL_NAME = "gemini-3.1-flash-lite"

os.makedirs(DOCUMENTS_DIR, exist_ok=True)

st.set_page_config(
    page_title="AI Research Assistant",
    page_icon="📚",
    layout="wide"
)


# --------------------------------
# 2. GEMINI CONFIGURATION
# --------------------------------

from google import genai

API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    st.error("Gemini API key not found. Configure GEMINI_API_KEY.")
    st.stop()

gemini_client = genai.Client(api_key=API_KEY)


# --------------------------------
# 3. MCP CONNECTION
# --------------------------------

server_params = StdioServerParameters(
    command=sys.executable,
    args=[SERVER_PATH],
    env=os.environ.copy()
)


async def call_mcp_tool(tool_name, arguments=None):

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                tool_name,
                arguments=arguments or {}
            )

            return "\n".join(
                item.text
                for item in result.content
                if item.type == "text"
            )


def run_mcp_tool(tool_name, arguments=None):
    return asyncio.run(
        call_mcp_tool(tool_name, arguments)
    )


# --------------------------------
# 4. APPLICATION HEADER
# --------------------------------

st.title("📚 AI Research Assistant")

st.caption(
    "Explore, summarise, compare and analyse "
    "research papers using MCP and Google Gemini."
)

st.divider()


# --------------------------------
# 5. SIDEBAR NAVIGATION
# --------------------------------

st.sidebar.title("Navigation")

page = st.sidebar.radio(
    "Choose a feature",
    [
        "Home",
        "Upload Papers",
        "Search Papers",
        "Summarise Paper",
        "Compare Papers",
        "Research Gaps",
        "Ask AI"
    ]
)


# --------------------------------
# 6. HOME PAGE
# --------------------------------

if page == "Home":

    st.header("Welcome to your Research Assistant")

    pdf_files = [
        f for f in os.listdir(DOCUMENTS_DIR)
        if f.lower().endswith(".pdf")
    ]

    col1, col2 = st.columns(2)

    col1.metric("Uploaded Research Papers", len(pdf_files))
    col2.metric("AI Model", MODEL_NAME)

    st.subheader("Available Features")

    st.markdown("""
    - 📄 Upload multiple research papers.
    - 🔍 Search across research documents.
    - 📝 Generate AI-powered summaries.
    - ⚖️ Compare research papers.
    - 💡 Identify potential research gaps.
    - 💬 Ask questions across multiple papers.
    """)


# --------------------------------
# 7. UPLOAD PAPERS
# --------------------------------

elif page == "Upload Papers":

    st.header("Upload Research Papers")

    uploaded_files = st.file_uploader(
        "Choose PDF files",
        type=["pdf"],
        accept_multiple_files=True
    )

    if st.button("Upload Documents"):

        if not uploaded_files:
            st.warning("Please select at least one PDF.")

        else:

            for uploaded_file in uploaded_files:

                file_path = os.path.join(
                    DOCUMENTS_DIR,
                    os.path.basename(uploaded_file.name)
                )

                with open(file_path, "wb") as file:
                    file.write(uploaded_file.getbuffer())

            st.success("Research papers uploaded successfully!")

    st.subheader("Available Documents")

    pdf_files = [
        f for f in os.listdir(DOCUMENTS_DIR)
        if f.lower().endswith(".pdf")
    ]

    for filename in pdf_files:
        st.write("📄", filename)


# --------------------------------
# 8. SEARCH PAPERS
# --------------------------------

elif page == "Search Papers":

    st.header("Search Research Papers")

    query = st.text_input(
        "Enter your search query",
        placeholder="Example: Deep learning models for brain tumour detection"
    )

    if st.button("Search"):

        if query.strip():

            with st.spinner("Searching research papers..."):

                results = run_mcp_tool(
                    "search_documents",
                    {"query": query}
                )

            st.subheader("Search Results")
            st.write(results)

        else:
            st.warning("Please enter a search query.")


# --------------------------------
# 9. SUMMARISE PAPER
# --------------------------------

elif page == "Summarise Paper":

    st.header("Research Paper Summarisation")

    pdf_files = [
        f for f in os.listdir(DOCUMENTS_DIR)
        if f.lower().endswith(".pdf")
    ]

    if pdf_files:

        filename = st.selectbox(
            "Select a research paper",
            pdf_files
        )

        if st.button("Generate Summary"):

            with st.spinner("Extracting and summarising paper..."):

                paper_text = run_mcp_tool(
                    "read_document",
                    {"filename": filename}
                )

                prompt = f"""
                You are an academic research assistant.

                Summarise this research paper using these headings:
                1. Research Objective
                2. Methodology
                3. Models and Algorithms
                4. Dataset
                5. Experimental Results
                6. Limitations
                7. Future Research Directions

                Rules:
                - Use only the supplied paper.
                - Do not invent information.
                - Write Not Reported if information is missing.
                - Distinguish survey findings from original experiments.

                PAPER:
                {paper_text}
                """

                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt
                )

            st.markdown(response.text)

    else:
        st.info("Upload research papers to get started.")


# --------------------------------
# 10. COMPARE PAPERS
# --------------------------------

elif page == "Compare Papers":

    st.header("Research Paper Comparison")

    pdf_files = [
        f for f in os.listdir(DOCUMENTS_DIR)
        if f.lower().endswith(".pdf")
    ]

    if len(pdf_files) >= 2:

        col1, col2 = st.columns(2)

        with col1:
            paper1 = st.selectbox(
                "Select Paper 1",
                pdf_files,
                key="paper1"
            )

        with col2:
            paper2 = st.selectbox(
                "Select Paper 2",
                pdf_files,
                index=1,
                key="paper2"
            )

        if st.button("Compare Papers"):

            if paper1 == paper2:
                st.warning("Please select two different papers.")

            else:

                with st.spinner("Comparing research papers..."):

                    summary1 = run_mcp_tool(
                        "read_document",
                        {"filename": paper1}
                    )

                    summary2 = run_mcp_tool(
                        "read_document",
                        {"filename": paper2}
                    )

                    prompt = f"""
                    Compare these two research papers.

                    PAPER 1 ({paper1}):
                    {summary1}

                    PAPER 2 ({paper2}):
                    {summary2}

                    Compare:
                    1. Research Objective
                    2. Methodology
                    3. Models and Algorithms
                    4. Dataset
                    5. Evaluation Metrics
                    6. Experimental Results
                    7. Strengths
                    8. Limitations
                    9. Key Differences
                    10. Overall Comparison

                    Rules:
                    - Use only supplied information.
                    - Do not invent results.
                    - Do not directly compare accuracy across
                      different datasets or evaluation tasks.
                    - Write Not Reported when information is missing.
                    """

                    response = gemini_client.models.generate_content(
                        model=MODEL_NAME,
                        contents=prompt
                    )

                st.markdown(response.text)

    else:
        st.info("Upload at least two research papers.")


# --------------------------------
# 11. RESEARCH GAP IDENTIFICATION
# --------------------------------

elif page == "Research Gaps":

    st.header("Research Gap Identification")

    pdf_files = [
        f for f in os.listdir(DOCUMENTS_DIR)
        if f.lower().endswith(".pdf")
    ]

    selected_papers = st.multiselect(
        "Select research papers",
        pdf_files
    )

    if st.button("Identify Research Gaps"):

        if not selected_papers:
            st.warning("Please select research papers.")

        else:

            with st.spinner("Analysing research papers..."):

                paper_context = ""

                for filename in selected_papers:

                    content = run_mcp_tool(
                        "read_document",
                        {"filename": filename}
                    )

                    paper_context += (
                        f"\n\nPAPER: {filename}\n{content}"
                    )

                prompt = f"""
                Analyse the following research papers.

                Identify:
                1. Common Limitations
                2. Paper-Specific Limitations
                3. Unresolved Research Problems
                4. Five Potential Research Gaps
                5. Possible Research Directions
                6. Three Potential Project Topics

                Rules:
                - Use only the supplied paper information.
                - Clearly label inferred research gaps.
                - Do not invent experimental results.
                - Distinguish author-reported limitations
                  from your own inferences.

                PAPERS:
                {paper_context}
                """

                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt
                )

            st.markdown(response.text)


# --------------------------------
# 12. ASK AI
# --------------------------------

elif page == "Ask AI":

    st.header("Ask Questions Across Research Papers")

    question = st.text_area(
        "Enter your question",
        placeholder="Example: Compare the architectures used in the brain tumour papers."
    )

    if st.button("Ask AI"):

        if question.strip():

            with st.spinner("Searching papers and generating answer..."):

                retrieved_context = run_mcp_tool(
                    "search_documents",
                    {"query": question}
                )

                prompt = f"""
                You are an academic research assistant.

                Answer the user's question using the retrieved
                research information.

                QUESTION:
                {question}

                RETRIEVED INFORMATION:
                {retrieved_context}

                Rules:
                - Use only the retrieved information.
                - Mention the PDF filename when possible.
                - Do not invent facts or results.
                - State when information is insufficient.
                """

                response = gemini_client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt
                )

            st.markdown(response.text)

        else:
            st.warning("Please enter a question.")

Writing /content/mcp_research_assistant/app.py


In [ ]:
import os

APP_PATH = "/content/mcp_research_assistant/app.py"

print("app.py exists:", os.path.exists(APP_PATH))
print("File size:", os.path.getsize(APP_PATH), "bytes")

app.py exists: True
File size: 12421 bytes


In [ ]:
import subprocess
import time
from google.colab.output import eval_js

APP_PATH = "/content/mcp_research_assistant/app.py"

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# Give Streamlit time to start
time.sleep(5)

if process.poll() is None:
    print("Streamlit application started!")
    url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("Open your application:", url)
else:
    print("Streamlit failed to start. Check the app.py file.")


Streamlit application started!
Open your application: https://8501-m-s-usw3b6-1p1tpljyxxrji-b.us-west3.prod.r.colab.dev


In [ ]:
import requests

try:
    response = requests.get(
        "http://localhost:8501/_stcore/health",
        timeout=10
    )
    print("Status code:", response.status_code)
    print("Response:", response.text)

except Exception as e:
    print("Error:", e)

Status code: 200
Response: ok


In [ ]:
import requests

response = requests.get("http://localhost:8501")
print("HTTP status:", response.status_code)
print("HTML received:", len(response.text), "characters")
print(response.text[:200])

HTTP status: 200
HTML received: 7863 characters
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the Lic


In [ ]:
import subprocess

result = subprocess.run(
    ["ps", "-ef"],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "streamlit" in line.lower():
        print(line)

root       43686   19686  3 13:40 ?        00:00:03 /usr/bin/python3 /usr/local/bin/streamlit run /content/mcp_research_assistant/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true


In [ ]:
import subprocess
import time
from google.colab.output import eval_js

# Stop the existing Streamlit process
process.terminate()
process.wait(timeout=10)

# Start Streamlit again
process = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/mcp_research_assistant/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Generate the Colab proxy URL
url = eval_js("google.colab.kernel.proxyPort(8501)")

print("Streamlit restarted successfully!")
print("Application URL:", url)

Streamlit restarted successfully!
Application URL: https://8501-m-s-usw3b6-1p1tpljyxxrji-b.us-west3.prod.r.colab.dev


In [ ]:
import subprocess
import time
import os

# Stop the previous process
process.terminate()
process.wait(timeout=10)

# Save logs to a file
log_file = open("/content/streamlit.log", "w")

# Start Streamlit again
process = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/mcp_research_assistant/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Process running:", process.poll() is None)

with open("/content/streamlit.log", "r") as f:
    print(f.read())

Process running: True


2026-11-09 13:44:10.182 Uvicorn server started on 0.0.0.0:8501



In [ ]:
%%writefile /content/test_app.py

import streamlit as st

st.set_page_config(
    page_title="Streamlit Test",
    page_icon="🚀"
)

st.title("Hello, Streamlit! 🚀")

st.success("Your Streamlit frontend is working.")

name = st.text_input("Enter your name")

if name:
    st.write(f"Welcome, {name}!")

st.balloons()

Writing /content/test_app.py


In [ ]:
import subprocess
import time
from google.colab.output import eval_js

# Start test application
test_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/test_app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Generate browser URL
test_url = eval_js(
    "google.colab.kernel.proxyPort(8502)"
)

print("Test process running:", test_process.poll() is None)
print("\nOpen this URL:")
print(test_url)

Test process running: True

Open this URL:
https://8502-m-s-usw3b6-1p1tpljyxxrji-b.us-west3.prod.r.colab.dev


In [ ]:
import requests

response = requests.get(
    "http://localhost:8502/_stcore/health",
    timeout=10
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: ok


In [ ]:
# Stop previous test server
test_process.terminate()
test_process.wait(timeout=10)

import subprocess
import time
from google.colab.output import eval_js

test_process = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/test_app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Process running:", test_process.poll() is None)

test_url = eval_js(
    "google.colab.kernel.proxyPort(8502)"
)

print("NEW TEST URL:")
print(test_url)

Process running: True
NEW TEST URL:
https://8502-m-s-usw3b6-1p1tpljyxxrji-b.us-west3.prod.r.colab.dev


In [ ]:
import subprocess
import time
from google.colab.output import eval_js

# Stop the existing main application
process.terminate()
process.wait(timeout=10)

# Restart with Colab-compatible settings
process = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/mcp_research_assistant/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(5)

app_url = eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("Your AI Research Assistant:")
print(app_url)

Your AI Research Assistant:
https://8501-m-s-usw3b6-1p1tpljyxxrji-b.us-west3.prod.r.colab.dev
